In [2]:
!git config --global user.email "beglossy@yonsei.ac.kr"
!git config --global user.name "beglossy-cmyk"

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
%cd gpt2.0_by_InaYoon
print("완료!")

Cloning into 'gpt2.0_by_InaYoon'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 23 (delta 10), reused 12 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 15.60 KiB | 7.80 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/gpt2.0_by_InaYoon
완료!


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

print(f"vocab_size: {vocab_size}")
print(f"xb.shape: {xb.shape}")

vocab_size: 65
xb.shape: torch.Size([64, 32])


In [4]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, head_size, block_size):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size):
        super().__init__()
        head_size = emb_dim // num_heads  # 64 // 4 = 16
        self.heads = nn.ModuleList([
            SingleHeadSelfAttention(emb_dim, head_size, block_size)
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(emb_dim, emb_dim)  # 합친 후 정리

    def forward(self, x):
        # 각 head 출력을 이어붙이기
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

# 테스트
mha = MultiHeadAttention(emb_dim=64, num_heads=4, block_size=32)
dummy = torch.randn(64, 32, 64)
out = mha(dummy)
print(f"multi-head attention 출력 shape: {out.shape}")  # (64, 32, 64)

multi-head attention 출력 shape: torch.Size([64, 32, 64])


In [5]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),  # 64 → 256
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),  # 256 → 64
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size):
        super().__init__()
        self.attn = MultiHeadAttention(emb_dim, num_heads, block_size)
        self.ff   = FeedForward(emb_dim)
        self.ln1  = nn.LayerNorm(emb_dim)  # 학습 안정화
        self.ln2  = nn.LayerNorm(emb_dim)

    def forward(self, x):
        # Residual connection: 원래 x를 더해줘서 학습 안정화
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# 테스트
block = Block(emb_dim=64, num_heads=4, block_size=32)
dummy = torch.randn(64, 32, 64)
out = block(dummy)
print(f"block 출력 shape: {out.shape}")  # (64, 32, 64)

block 출력 shape: torch.Size([64, 32, 64])


In [6]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64, num_heads=4, num_layers=4):
        super().__init__()
        self.token_embedding    = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size)
            for _ in range(num_layers)
        ])
        self.ln_f   = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        tok = self.token_embedding(x)
        pos = self.position_embedding(torch.arange(T, device=x.device))
        h = tok + pos
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

model = TinyGPT(vocab_size, block_size)
logits = model(xb)
print(f"logits.shape: {logits.shape}")  # (64, 32, 65)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f"총 파라미터 수: {total_params:,}")

logits.shape: torch.Size([64, 32, 65])
총 파라미터 수: 209,729


In [7]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 device: {device}")

model = TinyGPT(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

사용 device: cpu
epoch  0 | train loss 2.5642
epoch 10 | train loss 1.5712
epoch 20 | train loss 1.4877
epoch 30 | train loss 1.4542
epoch 40 | train loss 1.4298
epoch 50 | train loss 1.4118
epoch 60 | train loss 1.3925
epoch 70 | train loss 1.3866
epoch 80 | train loss 1.3787
epoch 90 | train loss 1.3702
epoch 99 | train loss 1.3647


In [8]:
@torch.no_grad()
def sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300))

ROMEO:
Measure you:
Yet have in? Good 'dragrance fight;
To save my properies, you for Even do in shall be the twolves through that;
For I have, in a
last, Edward drum.

GREMIO:
Why, neith, sir, I take
I had! what's to't.

GONZALO:
There, I am not?

LORD ROSS:
You suffer any lord!--
Dost thou thereby as mu


In [ ]:
from google.colab import userdata, _message
import json

nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)
with open('/content/gpt2.0_by_InaYoon/notebook_06_tiny_gpt.ipynb', 'w') as f:
    json.dump(nb['ipynb'], f)

token = userdata.get('GITHUB_TOKEN')
%cd /content/gpt2.0_by_InaYoon
!git remote set-url origin https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
!git add notebook_06_tiny_gpt.ipynb
!git commit -m "Add notebook_06: Tiny GPT"
!git push origin main